# SNNAI v6.0 Large-Scale SNN LM + Transformer Comparison

This notebook trains a large SNNAI character-level language model on a real corpus and compares it against a small Transformer baseline.

- v5.6: large-scale architecture
- v5.7: large-corpus training
- v5.8: Transformer comparison
- v5.9: quantization / pruning
- v6.0: Transformer-level challenge evaluation

In [ ]:
# Install dependencies
# CUDA 12.6 build keeps P100 (sm_60) and T4 (sm_75) compatibility
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install -q snntorch numpy requests

In [ ]:
# Clone SNNAI repository
!rm -rf snnai
!git clone --depth 1 --branch v6.0.6 https://github.com/sabunosuke1008-create/snnai.git
import sys
sys.path.insert(0, 'snnai')

In [ ]:
# Download tiny Shakespeare corpus
import requests, os, time, json
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
corpus = requests.get(url).text
print('Corpus length:', len(corpus))

In [ ]:
# Imports
import torch
import torch.nn as nn
from snnai.modules.language import CharTokenizer, SpikeEncoder
from snnai.modules.language.large_scale_lm import LargeScaleSNNLM, count_parameters
from snnai.benchmarks.transformer_comparison import TransformerBaseline, compare_models
from snnai.benchmarks.optimization import quantize_weights, prune_weights
from snnai.benchmarks.energy_quantification import quantize_energy

In [ ]:
# Tokenizer and dataset
vocab = sorted(set(corpus))
tokenizer = CharTokenizer(''.join(vocab))
data = torch.tensor([tokenizer.encode(corpus)], dtype=torch.long)
print('Vocab size:', tokenizer.vocab_size)
print('Data shape:', data.shape)

In [ ]:
# Training helpers
def make_batch(data, batch_size, seq_len, device):
    starts = torch.randint(0, data.size(1) - seq_len - 1, (batch_size,))
    inputs = torch.stack([data[0, s:s+seq_len] for s in starts])
    targets = torch.stack([data[0, s+1:s+seq_len+1] for s in starts])
    return inputs.to(device), targets.to(device)

def perplexity(logits, targets):
    loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
    return torch.exp(loss).item()

## Train large SNNAI model

In [ ]:
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    if cap[0] < 7:
        device = torch.device('cpu')
        print(f'GPU compute capability {cap} is below PyTorch minimum (7.0); using CPU')
    else:
        device = torch.device('cuda')
        print('Using device:', device)
else:
    device = torch.device('cpu')
    print('Using device:', device)

# Adjust dimensions for Kaggle T4; shrink on CPU fallback to fit timeout
if device.type == 'cpu':
    embed_dim = 64
    hidden_dim = 256
    num_layers = 2
    seq_len = 64
    batch_size = 16
    time_steps = 10
    epochs = 5
    print('Using small config for CPU fallback to avoid timeout')
else:
    embed_dim = 256
    hidden_dim = 1024
    num_layers = 4
    seq_len = 128
    batch_size = 32
    time_steps = 20
    epochs = 20
lr = 1e-3

snn_model = LargeScaleSNNLM(vocab_size=tokenizer.vocab_size, embed_dim=embed_dim,
                            hidden_dim=hidden_dim, num_layers=num_layers).to(device)
encoder = SpikeEncoder(vocab_size=tokenizer.vocab_size, time_steps=time_steps)
optimizer = torch.optim.Adam(snn_model.parameters(), lr=lr)
print('SNN params:', count_parameters(snn_model))

In [ ]:
snn_model.train()
snn_history = []
for epoch in range(epochs):
    inputs, targets = make_batch(data, batch_size, seq_len, device)
    spikes = encoder(inputs)  # (time_steps, batch, seq_len, vocab_size)
    optimizer.zero_grad()
    out = snn_model(spikes)  # (batch, seq_len, vocab_size)
    loss = nn.functional.cross_entropy(out.reshape(-1, tokenizer.vocab_size), targets.reshape(-1))
    loss.backward()
    optimizer.step()
    ppl = perplexity(out, targets)
    snn_history.append({'epoch': epoch, 'loss': loss.item(), 'ppl': ppl})
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch}: loss={loss.item():.3f} ppl={ppl:.2f}')

## Train Transformer baseline

In [ ]:
transformer_model = TransformerBaseline(vocab_size=tokenizer.vocab_size, d_model=256,
                                        nhead=4, num_layers=4, dim_feedforward=1024).to(device)
transformer_optimizer = torch.optim.Adam(transformer_model.parameters(), lr=lr)
transformer_history = []
for epoch in range(epochs):
    inputs, targets = make_batch(data, batch_size, seq_len, device)
    transformer_optimizer.zero_grad()
    out = transformer_model(inputs)
    loss = nn.functional.cross_entropy(out.reshape(-1, tokenizer.vocab_size), targets.reshape(-1))
    loss.backward()
    transformer_optimizer.step()
    ppl = perplexity(out, targets)
    transformer_history.append({'epoch': epoch, 'loss': loss.item(), 'ppl': ppl})
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f'Transformer epoch {epoch}: loss={loss.item():.3f} ppl={ppl:.2f}')

## Compare latency, parameters, and energy

In [ ]:
sample_input = torch.zeros(time_steps, 1, seq_len, tokenizer.vocab_size).to(device)
comparison = compare_models(snn_model, transformer_model, sample_input)
print(json.dumps(comparison, indent=2))

In [ ]:
energy_report = quantize_energy(snn_model, sample_input, time_steps=time_steps)
print('SNN energy report:')
print(json.dumps(energy_report, indent=2))

## Generate sample text

In [ ]:
snn_model.eval()
prompt = 'ROMEO:'
generated = prompt
with torch.no_grad():
    for _ in range(200):
        indices = torch.tensor([tokenizer.encode(generated[-seq_len:])], dtype=torch.long).to(device)
        one_hot = torch.zeros(1, indices.size(1), tokenizer.vocab_size).to(device)
        one_hot.scatter_(2, indices.unsqueeze(2), 1.0)
        x = one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1)
        out = snn_model(x)
        idx = int(out[0, -1].argmax().item())
        generated += tokenizer.idx_to_char[idx]
print('Generated:')
print(generated)

## Optimization: quantization and pruning

In [ ]:
scales = quantize_weights(snn_model)
print('Quantized scale sample:', list(scales.items())[:2])
prune_info = prune_weights(snn_model, threshold=0.005)
print('Pruning info:', prune_info)

## Save model

In [ ]:
torch.save({
    'model_state_dict': snn_model.state_dict(),
    'vocab': vocab,
    'history': snn_history,
    'comparison': comparison
}, 'snnai_v6_large_lm.pt')
print('Saved snnai_v6_large_lm.pt')